In [1]:
# !pip install -q transformers==4.57.2
# !pip install -q bitsandbytes trl

In [2]:
import torch
import os
import shutil
from huggingface_hub import snapshot_download
from transformers import AutoModelForImageTextToText, AutoProcessor 
from peft import PeftModel
# from transformers import AutoModelForImageTextToText, AutoProcessor 

In [3]:
# 1. Khai báo các đường dẫn
base_model_name = "Qwen/Qwen3-VL-2B-Instruct"
adapter_path = "/kaggle/input/models/npt2k5/qwen3-vl-2b-it-ft-ok/transformers/default/1"
save_path = "/kaggle/working/qwen3-vl-2b-it-finetuned-merged" 

print("Đang đồng bộ toàn bộ file phụ trợ từ repo gốc...")
# Tải (hoặc lấy từ cache) toàn bộ thư mục model gốc từ Hugging Face
base_model_path = snapshot_download(repo_id=base_model_name)

# Tạo thư mục lưu đích trước
os.makedirs(save_path, exist_ok=True)

# Copy TẤT CẢ các file cấu hình, vocab, merges... sang thư mục đích
for item in os.listdir(base_model_path):
    item_path = os.path.join(base_model_path, item)
    # Bỏ qua các file trọng số (weights) gốc vì chúng ta sẽ lưu file weights đã merge sau
    if os.path.isfile(item_path) and not item.endswith((".bin", ".safetensors", ".pt", ".h5", ".msgpack", ".index.json")):
        shutil.copy2(item_path, save_path)

print("Đang tải Base Model...")
# 2. Load mô hình gốc (Dùng AutoModelForImageTextToText để tránh lỗi Deprecated)
base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    _fast_init=False
).to("cuda")

print("Đang tải Processor (Tokenizer + Image Processor)...")
# 3. Load Processor
processor = AutoProcessor.from_pretrained(
    base_model_name, 
    trust_remote_code=True
)

print("Đang tải LoRA Adapter...")
# 4. Load LoRA adapter đắp lên base model
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Đang tiến hành Merge (gộp) trọng số...")
# 5. Gộp trọng số của LoRA vào mô hình gốc và giải phóng bộ nhớ PEFT
merged_model = model.merge_and_unload()

merged_model = merged_model.to(torch.bfloat16) 
merged_model.eval()

print(f"Đang lưu mô hình mới vào: {save_path}")
# 6. Lưu mô hình và PROCESSOR (Sẽ tự ghi đè lên các file config và sinh ra model.safetensors mới)
merged_model.save_pretrained(save_path, safe_serialization=True)
processor.save_pretrained(save_path) 

print("Hoàn tất! Bạn có thể sử dụng mô hình trong thư mục save_path.")

Đang đồng bộ toàn bộ file phụ trợ từ repo gốc...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Đang tải Base Model...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Đang tải Processor (Tokenizer + Image Processor)...
Đang tải LoRA Adapter...
Đang tiến hành Merge (gộp) trọng số...
Đang lưu mô hình mới vào: /kaggle/working/qwen3-vl-2b-it-finetuned-merged


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Hoàn tất! Bạn có thể sử dụng mô hình trong thư mục save_path.
